# 第24章 窗口计算与探索性分析

结合滚动窗口、累计指标和描述性统计完成探索性分析。


## 先解决一个小问题

拿一组小型业务数据练习“窗口计算与探索性分析”：先看数据结构，再完成一次明确的计算或转换。结合滚动窗口、累计指标和描述性统计完成探索性分析。


## 这章为什么先学

这是“Pandas”路线中第 24 章的操作重点。本章只解决“窗口计算与探索性分析”，不重复前面章节已经完成的准备工作。


## 开始前确认

- 掌握 Python 基础语法、列表和字典
- 开始前先确认：计算滚动与累计指标


## 做完要留下什么

产出一个与“窗口计算与探索性分析”直接对应的结果，并记录输入形状、字段或筛选口径。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 计算滚动与累计指标
- 生成描述性统计
- 检查频数和分位数
- 分析相关关系但避免因果误读


## 核心概念

- 窗口大小必须与业务周期一致。
- 滚动统计前要按时间排序。
- 相关系数只描述线性共同变化，不能证明因果。


## 示例 1：滚动与累计计算

min_periods控制窗口不足时是否返回结果。


In [ ]:
import pandas as pd

sales = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=10, freq="D"),
    "amount": [120, 135, 128, 160, 175, 168, 190, 205, 198, 220],
})
sales["rolling_3d"] = sales["amount"].rolling(3, min_periods=1).mean()
sales["cumulative"] = sales["amount"].cumsum()
sales["daily_growth"] = sales["amount"].pct_change()
print(sales.round(3))


## 示例 2：描述性统计与频数

数值概览和类别频数应一起检查。


In [ ]:
orders = pd.DataFrame({
    "region": ["华东", "华南", "华东", "华北", "华东", "华南"],
    "amount": [320, 880, 460, 1250, 720, 540],
    "items": [2, 4, 1, 5, 3, 2],
})
print(orders[["amount", "items"]].describe().round(2))
print(orders["region"].value_counts(normalize=True).round(3))


## 示例 3：相关与分位数

先检查散点和异常，再解释相关系数。


In [ ]:
print("分位数:\n", orders["amount"].quantile([0.25, 0.5, 0.75]))
print("相关矩阵:\n", orders[["amount", "items"]].corr().round(3))
orders["amount_rank"] = orders["amount"].rank(ascending=False, method="dense")
print(orders.sort_values("amount_rank"))


## 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd
from js import window

# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = f"{window.location.origin}/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates=["InvoiceDate"],
    dtype={"InvoiceNo": "string", "StockCode": "string", "Description": "string", "Country": "category"},
).rename(columns={
    "InvoiceNo": "order_id", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "order_time", "UnitPrice": "unit_price",
    "CustomerID": "customer_id", "Country": "country",
})
large_orders["sales"] = (large_orders["quantity"] * large_orders["unit_price"]).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C") | (large_orders["quantity"] < 0),
    "取消/退货", "完成"
)
print(f"UCI Online Retail 公开数据：{len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print("内存占用：", f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
large_orders.head()


In [ ]:
daily_sales = (
    large_orders.query("status == '完成'")
    .set_index("order_time")["sales"]
    .resample("D").sum()
    .to_frame("sales")
)
daily_sales["rolling_7d"] = daily_sales["sales"].rolling(7, min_periods=1).mean()
daily_sales["rolling_30d"] = daily_sales["sales"].rolling(30, min_periods=7).mean()
daily_sales["growth_7d"] = daily_sales["sales"].pct_change(7)
daily_sales["z_score"] = (daily_sales["sales"] - daily_sales["sales"].mean()) / daily_sales["sales"].std()
print(f"从 {len(large_orders):,} 笔订单得到 {len(daily_sales):,} 天趋势")
display(daily_sales.tail(10).round(3))
display(daily_sales.nlargest(5, "z_score").round(2))


## 常见误区

- 未按日期排序就计算滚动窗口
- 窗口长度与业务周期不一致
- 把相关系数解释为因果效应


## 综合练习

1. 创建12个月销售序列
2. 计算3个月移动平均和累计销售
3. 找出销售额最高的3个月

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“创建12个月销售序列”。
2. **独立完成**：不复制示例代码，完成“计算3个月移动平均和累计销售”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“找出销售额最高的3个月”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import pandas as pd

monthly = pd.DataFrame({
    "month": pd.period_range("2025-01", periods=12, freq="M").astype("string"),
    "sales": [120, 128, 135, 142, 150, 146, 160, 172, 180, 188, 205, 218],
})

# TODO: 计算3个月移动平均
monthly["moving_3m"] =

# TODO: 计算累计销售
monthly["cumulative"] =

print(monthly.round(2))
print("Top 3:\n", monthly.nlargest(3, "sales")[["month", "sales"]])


In [ ]:
import pandas as pd

monthly = pd.DataFrame({
    "month": pd.period_range("2025-01", periods=12, freq="M").astype("string"),
    "sales": [120, 128, 135, 142, 150, 146, 160, 172, 180, 188, 205, 218],
})
monthly["moving_3m"] = monthly["sales"].rolling(3, min_periods=1).mean()
monthly["cumulative"] = monthly["sales"].cumsum()
print(monthly.round(2))
print("Top 3:\n", monthly.nlargest(3, "sales")[["month", "sales"]])

# 自检
assert monthly["cumulative"].iloc[-1] == 1944, "检查累计销售：最后一行应该是总和"
assert len(monthly.nlargest(3, "sales")) == 3, "检查Top 3行数"


## 本章小结

结合滚动窗口、累计指标和描述性统计完成探索性分析。

**迁移思考**：

1. 如果需要计算7日移动平均但前6天数据不足，min_periods 应该设置为多少？
2. 为什么相关系数高不代表因果关系？请举一个相关但无因果的例子。


### 你已经掌握

- 计算滚动与累计指标
- 生成描述性统计
- 检查频数和分位数
- 分析相关关系但避免因果误读


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 滚动与累计计算 | min_periods控制窗口不足时是否返回结果。 | `pd.DataFrame()`、`pd.date_range()`、`sales.round()`、`.rolling()` |
| 描述性统计与频数 | 数值概览和类别频数应一起检查。 | `pd.DataFrame()`、`.describe()`、`.round()`、`.value_counts()` |
| 相关与分位数 | 先检查散点和异常，再解释相关系数。 | `orders.sort_values()`、`.quantile()`、`.corr()`、`.round()` |


### 需要注意

- 未按日期排序就计算滚动窗口
- 窗口长度与业务周期不一致
- 把相关系数解释为因果效应


### 完成检查

- [ ] 能够计算滚动与累计指标
- [ ] 能够生成描述性统计
- [ ] 能够检查频数和分位数
- [ ] 能够分析相关关系但避免因果误读


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
